<div style="text-align: center;">
<a target="_blank" href="https://colab.research.google.com/github/miquelmn/aa_2627/blob/main/06_Transfer/AlexNet_Transfer.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
</div>

In [2]:
import torch
from torchvision import datasets, models, transforms

# Models ja existents i *transfer learning*

En aquesta pràctica, aprofundirem en la classificació d’imatges amb xarxes neuronals convolucionals (CNNs), però amb un enfocament diferent al de la sessió anterior. Mentre que anteriorment vàrem construir CNNs des de zero per comprendre la seva estructura bàsica, aquest cop treballarem amb un model de CNN preentrenat: **AlexNet**.

### AlexNet


**AlexNet** és l'arquitectura de xarxa neuronal convolucional (CNN) dissenyada per Alex Krizhevsky, Ilya Sutskever i Geoffrey Hinton (Universitat de Toronto) que va transformar el camp de la visió per computador l'any 2012. En guanyar la competició **ILSVRC-2012 (ImageNet Large Scale Visual Recognition Challenge)** amb un error *top-5* del **15,3%** —enfront del 26,2% obtingut pel segon classificat, basat en mètodes clàssics—, va marcar el punt d'inflexió decisiu cap a l'ús massiu de xarxes neuronals profundes.

![Arquitectura](https://viso.ai/wp-content/uploads/2024/04/alexNet-architecture.png)

#### 1. Context i Impacte Històric

Abans d'AlexNet, la visió per computador depenia principalment de mètodes en dues etapes:
* **Extracció artesanal de característiques (*hand-crafted features*):** Algoritmes com SIFT, SURF o HOG dissenyats manualment per experts.
* **Classificadors superficials:** Models com Support Vector Machines (SVM) o Random Forests alimentats amb aquests descriptors.

AlexNet va demostrar empíricament la superioritat d'un nou paradigma:
1. **Aprenentatge extrem a extrem (*end-to-end*):** Els filtres i les jerarquies visuals s'aprenen automàticament des dels píxels bruts.
2. **Acceleració massiva en GPU:** L'ús de maquinari paral·lel va permetre entrenar milions de paràmetres en dies en lloc de setmanes o mesos.
3. **Escala massiva de dades:** L'ús de la base de dades ImageNet (més d'1,2 milions d'imatges d'entrenament repartides en 1.000 classes) va permetre generalitzar models amb gran capacitat sense sobreajustament immediat.

#### 2. Arquitectura de la Xarxa

AlexNet conté aproximadament **60 milions de paràmetres** i **650.000 neurones**, distribuïts al llarg de **8 capes d'aprenentatge amb pesos**: 5 capes convolucionals i 3 capes completament connectades (*fully connected* o FC).

| Capa | Tipus | Filtres / Mida del nucli (*Kernel*) | Pas (*Stride*) | Mida del mapa de sortida |
| :--- | :--- | :--- | :--- | :--- |
| **Entrada** | Imatge RGB | — | — | $224 \times 224 \times 3$ |
| **Conv 1** | Convolució + MaxPool + LRN | 96 filtres d'$11 \times 11$ | 4 | $55 \times 55 \times 96 \rightarrow 27 \times 27 \times 96$ |
| **Conv 2** | Convolució + MaxPool + LRN | 256 filtres de $5 \times 5$ | 1 (pad 2) | $27 \times 27 \times 256 \rightarrow 13 \times 13 \times 256$ |
| **Conv 3** | Convolució | 384 filtres de $3 \times 3$ | 1 (pad 1) | $13 \times 13 \times 384$ |
| **Conv 4** | Convolució | 384 filtres de $3 \times 3$ | 1 (pad 1) | $13 \times 13 \times 384$ |
| **Conv 5** | Convolució + MaxPool | 256 filtres de $3 \times 3$ | 1 (pad 1) | $13 \times 13 \times 256 \rightarrow 6 \times 6 \times 256$ |
| **FC 6** | Densa + Dropout ($p=0.5$) | 4.096 neurones | — | 4.096 |
| **FC 7** | Densa + Dropout ($p=0.5$) | 4.096 neurones | — | 4.096 |
| **FC 8** | Densa (Sortida) | 1.000 classes (Softmax) | — | 1.000 |

---

### Transfer Learning?

El **Transfer Learning** és una tècnica d'aprenentatge profund que permet aprofitar xarxes neuronals prèviament entrenades en conjunts de dades massius (com ara **AlexNet** entrenada sobre ImageNet) per resoldre una tasca nova o específica.

En lloc d'iniciar l'entrenament des de zero amb pesos aleatoris —procés que requereix grans volums de dades i un elevat cost computacional—, adaptem l'arquitectura a les noves classes del nostre conjunt de dades. D'aquesta manera, el nou model es beneficia directament de les representacions jeràrquiques ja apreses per la xarxa base (patrons visuals universals com ara vores, textures, gradients o formes geomètriques bàsiques).

## Estratègies d'Adaptació

Segons el volum de dades disponible i la similitud entre la tasca d'origen i la nova tasca, aquest procés es pot dur a terme principalment mitjançant dues estratègies:

#### 1. *Transfer Learning* com a Extractor de Característiques Fix (*Feature Extraction*)

En aquest enfocament, s'assumeix que les característiques apreses per les capes convolucionals són suficients per a la nova tasca:

* **Congelació de l'extractor:** Es fixen els pesos de les capes convolucionals (`features`) desactivant el càlcul de gradients. D'aquesta manera, els pesos originals no es modifiquen durant la retropropagació.
* **Nou classificador:** Se substitueix la capa de sortida original (com ara el bloc fully-connected o `MLP`) per un nou capçal adaptat al nombre de classes de la nova tasca, el qual s'entrena completament des de zero.

```python
# Congelació de tots els paràmetres de l'extractor de característiques
for param in alex.features.parameters():
    param.requires_grad = False

# Adaptació del classificador per a la nova tasca (p. ex., 10 classes)
import torch.nn as nn
num_ftrs = alex.classifier[6].in_features
alex.classifier[6] = nn.Linear(num_ftrs, 10)
```

---

#### 2. *Fine-Tuning*

El **fine-tuning** consisteix a permetre que part de la xarxa preentrenada continuï aprenent per adaptar-se millor a les particularitats del nou domini:

* **Congelació parcial:** Es mantenen fixes les primeres capes del model (on es capturen els patrons més generals com línies i colors), però es descongelen les darreres capes convolucionals, més especialitzades en formes d'alt nivell.
* **Reentrenament conjunt:** Es reentrenen tant el nou classificador com el subconjunt seleccionat de capes convolucionals.
* **Taxes d'aprenentatge diferenciades (*Differential Learning Rates*):** Per evitar destruir el coneixement previ (*catastrophic forgetting*), s'utilitza una taxa d'aprenentatge (*learning rate*) substancialment **més petita** per a les capes preentrenades que per al nou classificador, utilitzant grups de paràmetres a l'optimitzador:

```python
import torch.optim as optim

# Configuració de taxes d'aprenentatge diferenciades per grup de capes
optimizer = optim.Adam([
    {"params": model.features[8:].parameters(), "lr": 1e-5},  # Capes convolucionals profundes
    {"params": model.classifier.parameters(),   "lr": 1e-3},  # Classificador nou
])
```

## Començam

Primer de tot, com sempre, hem d'obtenir les dades. Aquesta sessió la farem amb el conjunt de dades [Tiny ImageNet](https://www.kaggle.com/c/tiny-imagenet/data?select=train.images.zip).

Aquest conjunt de dades es defineix en la seva plana de la forma següent:

> MicroImageNet classification challenge is similar to the classification challenge in the full ImageNet ILSVRC. MicroImageNet contains 200 classes for training. Each class has 500 images. The test set contains 10,000 images. All images are 64x64 colored ones.

Aquesta vegada no farem feina amb un conjunt de dades ja existents a ``torchvision`` sinó que nosaltres farem la gestió des de 0. Per tant i primer de tot descarregarem les dades. Per fer-ho podem trobar el conjunt de dades a la següent plana (http://cs231n.stanford.edu/tiny-imagenet-200.zip).

Alternativament també podem emprar l'eina ``wget`` així:

```
wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
```

Una vegada que hem descarregat les dades les podem descomprimir i finalment comença a fer-hi feina.

In [3]:
# Descargar y descomprimir Tiny ImageNet directamente en Colab
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip -q tiny-imagenet-200.zip

--2026-10-21 09:29:52--  http://cs231n.stanford.edu/tiny-imagenet-200.zip
Resolving cs231n.stanford.edu (cs231n.stanford.edu)... 171.64.64.64
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://cs231n.stanford.edu/tiny-imagenet-200.zip [following]
--2026-10-21 09:29:52--  https://cs231n.stanford.edu/tiny-imagenet-200.zip
Connecting to cs231n.stanford.edu (cs231n.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 248100043 (237M) [application/zip]
Saving to: ‘tiny-imagenet-200.zip’

tiny-imagenet-200.z 100%[===================>] 236.61M  15.5MB/s    in 14s     

2026-10-21 09:30:06 (17.5 MB/s) - ‘tiny-imagenet-200.zip’ saved [248100043/248100043]



In [4]:
import os
import shutil

val_dir = 'tiny-imagenet-200/val'
img_dir = os.path.join(val_dir, 'images')
annot_file = os.path.join(val_dir, 'val_annotations.txt')

# Leer el archivo de anotaciones y mover cada imagen a la subcarpeta de su clase
if os.path.exists(img_dir):
    with open(annot_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            img_name, class_id = parts[0], parts[1]

            class_dir = os.path.join(val_dir, class_id)
            os.makedirs(class_dir, exist_ok=True)

            src = os.path.join(img_dir, img_name)
            dst = os.path.join(class_dir, img_name)
            if os.path.exists(src):
                shutil.move(src, dst)

    # Eliminar la carpeta 'images' original una vez vacía
    shutil.rmtree(img_dir)
    print("Carpeta 'val' organizada correctamente por clases.")

Carpeta 'val' organizada correctamente por clases.


In [5]:
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Rutas corregidas para Google Colab
train = datasets.ImageFolder('tiny-imagenet-200/train', transform=transform)
test = datasets.ImageFolder('tiny-imagenet-200/val', transform=transform)

train_loader = torch.utils.data.DataLoader(train,
                                           batch_size=BATCH_SIZE,
                                           shuffle=True,
                                           num_workers=NUM_WORKERS)
test_loader = torch.utils.data.DataLoader(test,
                                          batch_size=BATCH_SIZE,
                                          shuffle=False,
                                          num_workers=NUM_WORKERS)

In [6]:
img, target = next(iter(train_loader))
print(img.shape, target)

torch.Size([64, 3, 224, 224]) tensor([ 26, 127,  89, 176, 116,  13, 113, 194,  73, 129, 114, 193, 183,  99,
         98,  36, 129,  38, 174,  40, 193,  71,  81, 109,  61,  27, 125,  75,
        110, 107, 160, 135,  83, 171,  62, 117,  15,  68,  18, 164,  74,  46,
        144,  13,  58,  39, 157, 137, 187, 174,   9,  77,  46,   1,  21,  10,
         29,  10, 185,  88,  61,  41, 157, 133])


## Definició de la xarxa: AlexNet i *Transfer learning*

En aquesta pràctica aplicarem la tècnica de transfer learning amb la primera xarxa CNN moderna:
- AlexNet. [ImageNet Classification with Deep Convolutional Neural Network, 2012](https://proceedings.neurips.cc/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf). La mida d'entrada de les imatges és de (227x227x3).Té prop de 60 milions de paràmetres entrenables.

Pytorch ens permet emprar aquest tipus de xarxes de manera molt senzilla. [Més informació](https://pytorch.org/vision/stable/models.html). Si el model que cercam no es troba integrat dins la llibreria Pytorch és bastant probable que si la trobem a Huggingface.

Descarregarem AlexNet i a analitzar-la. En aquest cas no només ens baixam la seva arquitectura, també els pesos resultants de l'entrenament.

**Normalment els problems els resoldrem emprant models ja definits i preentrenats**

In [7]:
alex = models.alexnet(weights=True)

print("-" * 50)
print("Arquitectura AlexNet")
print("-" * 50)
print(alex)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 166MB/s]


--------------------------------------------------
Arquitectura AlexNet
--------------------------------------------------
AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classi

Podem accedir a una capa concreta pel seu nom o índex dins de l’estructura del model. Per exemple:

In [8]:
alex.features[0]

Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))

El que nosaltres volem fer és emprar els pesos ja entrenats d'aquest model i aplicar-ho per resoldre un problema nou. Hi ha diverses maneres de realitzar la tècnica de ``Transfer Learning``. Les dues les hem explicades a la introducció, en aquest cas però emprarem el ``transfer learning`` pròpiament dit. L'element principal és congelar les capes de l'extractor de característiques. Per fer-ho empram les següents instruccions

```
for param in alex.features.parameters():
   param.requires_grad = False

```

## Feina a fer:

1. Carregar la xarxa AlexNet:
   1. Emprar el model sense reentrenar.
   2. Entrenar el model emprant *fine-tunning*.
   3. Entrenar el model emprant *transfer learning*.
2. Avaluar els diferents models.
3. Guardau els millors pesos de les vostres xarxes i tornau-los a carregar. Classificar una imatge del conjunt de test.

**Nota**. Com veureu no us donam aquesta vegada el bucle d'entrenament, sigui com sigui podeu adaptar el vist a les sessions anteriors.


## EXTRA: Com emprar la GPU per entrenar un model

Un dels elements diferencials d'aquest model, respecte als que havíem vist fins ara, és la seva mida i, per tant, l'entrenament es torna impossible emprant __CPU__ directament. Per resoldre-ho hem d'emprar una **GPU**, a Google Colab disposam d'elles gratuïtament. Per fer-ho amb *Pytorch* hem de fer tres passes:

1. Comprovar que hi ha una GPU disponible.
2. Moure el model a GPU.
3. Moure les dades a GPU.

### Comprova si tenim una GPU disponible

Primer de tot, cal verificar si hi ha una GPU disponible a l’entorn. Això es pot fer amb el següent codi:

```python

import torch

is_cuda = torch.cuda.is_available()
```

Si la variable ``is_cuda`` és certa, llavors tens accés a una GPU.

### Mou el model a la GPU

En PyTorch, els models han d'estar explícitament en la GPU per poder fer servir la seva potència de càlcul. Si estàs carregant un model preentrenat (com AlexNet, ResNet, etc.), o si has definit el teu propi model, pots moure’l a la GPU amb ``.to(device)``, on device fa referència a la GPU.

```python

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
```

Això mou el model a la GPU (si està disponible). Si només tens una CPU, el model es mantindrà a la CPU.

### Mou les dades a la GPU

No només el model, sinó que també les dades (inputs) han d'estar a la GPU per fer les operacions més ràpides. Així, abans de fer servir les dades com a inputs del model, assegura't de moure-les al mateix device:

```python

# Exemple d'un batch de dades
inputs, labels = inputs.to(device), labels.to(device)
```

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import copy

# 1. Comprobar y configurar la GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

NUM_CLASSES = 200 # Tiny ImageNet tiene 200 clases
criterion = nn.CrossEntropyLoss()

# Función para evaluar cualquier modelo en el conjunto de test/validación
def evaluar_modelo(model, dataloader):
    model.eval()
    corrects = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * inputs.size(0)
            corrects += torch.sum(preds == labels.data)
            total += labels.size(0)

    loss_final = running_loss / total
    acc_final = corrects.double() / total
    return loss_final, acc_final.item()

# Función de entrenamiento que guarda los mejores pesos
def entrenar_modelo(model, train_loader, test_loader, optimizer, epochs=5, nombre_archivo="mejor_modelo.pth"):
    mejor_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        corrects = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            running_loss += loss.item() * inputs.size(0)
            corrects += torch.sum(preds == labels.data)
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = (corrects.double() / total).item()

        # Evaluar al final de cada época
        val_loss, val_acc = evaluar_modelo(model, test_loader)

        print(f"Época {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc*100:.2f}%")

        # Guardar los pesos si obtenemos el mejor resultado hasta ahora
        if val_acc > mejor_acc:
            mejor_acc = val_acc
            torch.save(model.state_dict(), nombre_archivo)

    print(f"--> Mejor Val Accuracy: {mejor_acc*100:.2f}% (Pesos guardados en '{nombre_archivo}')\n")
    return mejor_acc

Usando dispositivo: cuda


In [ ]:
# ==========================================
# 1.1. MODELO SIN REENTRENAR
# ==========================================
print("--- 1.1 Evaluando AlexNet SIN reentrenar ---")
alex_sin_entrenar = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

# Adaptamos la última capa a las 200 clases de Tiny ImageNet
num_ftrs = alex_sin_entrenar.classifier[6].in_features
alex_sin_entrenar.classifier[6] = nn.Linear(num_ftrs, NUM_CLASSES)
alex_sin_entrenar = alex_sin_entrenar.to(device)

loss_0, acc_0 = evaluar_modelo(alex_sin_entrenar, test_loader)
print(f"Resultado sin entrenar -> Val Loss: {loss_0:.4f} | Val Acc: {acc_0*100:.2f}%\n")


# ==========================================
# 1.3. MODELO CON TRANSFER LEARNING (Feature Extraction)
# ==========================================
print("--- 1.3 Entrenando AlexNet con Transfer Learning (Extractor congelado) ---")
alex_tl = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

# 1. Congelamos los pesos del extractor de características
for param in alex_tl.features.parameters():
    param.requires_grad = False

# 2. Cambiamos la última capa para 200 clases
num_ftrs = alex_tl.classifier[6].in_features
alex_tl.classifier[6] = nn.Linear(num_ftrs, NUM_CLASSES)
alex_tl = alex_tl.to(device)

# 3. Optimizamos solo los parámetros del clasificador
optimizer_tl = optim.Adam(alex_tl.classifier.parameters(), lr=1e-3)

acc_tl = entrenar_modelo(alex_tl, train_loader, test_loader, optimizer_tl,
                         epochs=5, nombre_archivo="mejores_pesos_tl.pth")

In [10]:
# ==========================================
# 1.2. MODELO CON FINE-TUNING
# ==========================================
print("--- 1.2 Entrenando AlexNet con Fine-Tuning ---")
alex_ft = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

# 1. Congelación parcial: congelamos solo las primeras capas (hasta el índice 7)
for param in alex_ft.features[:8].parameters():
    param.requires_grad = False

# 2. Adaptamos la capa de salida a 200 clases
num_ftrs = alex_ft.classifier[6].in_features
alex_ft.classifier[6] = nn.Linear(num_ftrs, NUM_CLASSES)
alex_ft = alex_ft.to(device)

# 3. Tasas de aprendizaje diferenciadas (Differential Learning Rates)
optimizer_ft = optim.Adam([
    {"params": alex_ft.features[8:].parameters(), "lr": 1e-5}, # Capas convolucionales profundas
    {"params": alex_ft.classifier.parameters(),   "lr": 1e-4}, # Clasificador
])

acc_ft = entrenar_modelo(alex_ft, train_loader, test_loader, optimizer_ft,
                         epochs=5, nombre_archivo="mejores_pesos_ft.pth")

# ==========================================
# 2. RESUMEN DE LA EVALUACIÓN DE LOS 3 MODELOS
# ==========================================
print("-" * 50)
print("RESUMEN DE RESULTADOS (Accuracy en Test):")
print(f"1. Sin reentrenar:    {acc_0*100:.2f}%")
print(f"2. Transfer Learning: {acc_tl*100:.2f}%")
print(f"3. Fine-Tuning:       {acc_ft*100:.2f}%")
print("-" * 50)

--- 1.2 Entrenando AlexNet con Fine-Tuning ---
Época 1/5 | Train Loss: 2.3414 - Train Acc: 43.79% | Val Loss: 1.8294 - Val Acc: 53.90%
Época 2/5 | Train Loss: 1.5863 - Train Acc: 58.79% | Val Loss: 1.6907 - Val Acc: 57.43%
Época 3/5 | Train Loss: 1.2153 - Train Acc: 67.01% | Val Loss: 1.6691 - Val Acc: 57.61%
Época 4/5 | Train Loss: 0.9139 - Train Acc: 74.26% | Val Loss: 1.6930 - Val Acc: 57.74%
Época 5/5 | Train Loss: 0.6787 - Train Acc: 80.20% | Val Loss: 1.7467 - Val Acc: 57.46%
--> Mejor Val Accuracy: 57.74% (Pesos guardados en 'mejores_pesos_ft.pth')

--------------------------------------------------
RESUMEN DE RESULTADOS (Accuracy en Test):


NameError: name 'acc_0' is not defined

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Definir la arquitectura vacía y cargar los mejores pesos guardados
modelo_final = models.alexnet(weights=None)
modelo_final.classifier[6] = nn.Linear(modelo_final.classifier[6].in_features, NUM_CLASSES)

# Cargamos los mejores pesos (por ejemplo, los de Fine-Tuning)
archivo_pesos = "mejores_pesos_ft.pth" if acc_ft >= acc_tl else "mejores_pesos_tl.pth"
modelo_final.load_state_dict(torch.load(archivo_pesos, map_location=device))
modelo_final = modelo_final.to(device)
modelo_final.eval()
print(f"Pesos cargados correctamente desde: {archivo_pesos}")

# 2. Obtener una imagen del conjunto de test y predecir su clase
img_tensor, label_idx = test[0] # Puedes cambiar el 0 por cualquier número entre 0 y 9999
input_batch = img_tensor.unsqueeze(0).to(device) # Añadimos dimensión de batch (1, 3, 224, 224)

with torch.no_grad():
    salida = modelo_final(input_batch)
    probabilidades = torch.nn.functional.softmax(salida[0], dim=0)
    confianza, pred_idx = torch.max(probabilidades, 0)

clase_real = test.classes[label_idx]
clase_predicha = test.classes[pred_idx.item()]

# 3. Des-normalizar la imagen para visualizarla correctamente con Matplotlib
img_mostrar = img_tensor.cpu().numpy().transpose((1, 2, 0))
img_mostrar = np.array(IMAGENET_STD) * img_mostrar + np.array(IMAGENET_MEAN)
img_mostrar = np.clip(img_mostrar, 0, 1)

plt.imshow(img_mostrar)
plt.axis('off')
plt.title(f"Real: {clase_real} | Predicción: {clase_predicha} ({confianza.item()*100:.1f}%)")
plt.show()